# 15. 3Sum

[Link to Problem](https://leetcode.com/problems/3sum/)

### Description

Given an integer array nums, return all the triplets `[nums[i], nums[j], nums[k]]` such that `i != j`, `i != k`, and `j != k`, and `nums[i] + nums[j] + nums[k] == 0`.

Notice that the solution set must not contain duplicate triplets.

**Example 1:**
```
Input: nums = [-1,0,1,2,-1,-4]
Output: [[-1,-1,2],[-1,0,1]]
Explanation: 
nums[0] + nums[1] + nums[2] = (-1) + 0 + 1 = 0.
nums[1] + nums[2] + nums[4] = 0 + 1 + (-1) = 0.
nums[0] + nums[3] + nums[4] = (-1) + 2 + (-1) = 0.
The distinct triplets are [-1,0,1] and [-1,-1,2].
Notice that the order of the output and the order of the triplets does not matter.
```

**Example 2:**
```
Input: nums = [0,1,1]
Output: []
Explanation: The only possible triplet does not sum up to 0.
```

**Example 3:**
```
Input: nums = [0,0,0]
Output: [[0,0,0]]
Explanation: The only possible triplet sums up to 0.
```

**Constraints:**
- `3 <= nums.length <= 3000`
- `-10^5 <= nums[i] <= 10^5`

## My Intuition

_Write down your thoughts, ideas, and insights here._

- **Observations:**
- **Edge cases:**
  - 
- **Expected approach and complexity:**
  1. Time:  O(n*n)
  2. Space: O(n)

In [37]:
# my solution: not optimal
def threeSum(nums: list[int]) -> list[list[int]]:

    size = len(nums)
    
    def twoSum(start) -> list[list[int]]:
        target = nums[start]
        triplets = []
        seen = set()
        for i in range(size-start-1):          
            num = nums[start+i+1]
            complement = -(target + num)
            if complement in seen:
                triplet = [target, complement, num]
                triplets.append(triplet)
            seen.add(num)
            
        return triplets

    def remove_duplicate_arrays(data):
        seen = set()
        result = []
        for arr in data:
            key = tuple(sorted(arr))
            if key not in seen:
                seen.add(key)
                result.append(arr)
        return result
        
    triplets = []
    for i in range(size):
        payload = twoSum(i)
        triplets.extend(payload)

    unique_triplets = remove_duplicate_arrays(triplets)
        
    return unique_triplets


if __name__ == '__main__':
    # Test Case 1
    nums1 = [-1, 0, 1, 2, -1, -4]
    result1 = threeSum(nums1)
    print(f"Test 1 Input: nums = {nums1}")
    print(f"Test 1 Output: {result1}") # Expected: [[-1, -1, 2], [-1, 0, 1]]

    # Test Case 2
    nums2 = [0, 1, 1]
    result2 = threeSum(nums2)
    print(f"\nTest 2 Input: nums = {nums2}")
    print(f"Test 2 Output: {result2}") # Expected: []

    # Test Case 3
    nums3 = [0, 0, 0]
    result3 = threeSum(nums3)
    print(f"\nTest 3 Input: nums = {nums3}")
    print(f"Test 3 Output: {result3}") # Expected: [[0, 0, 0]]

    # Test Case 4
    nums4 = [2, -4]
    result4 = threeSum(nums4)
    print(f"\nTest 4 Input: nums = {nums4}")
    print(f"Test 4 Output: {result4}") # Expected: []

Test 1 Input: nums = [-1, 0, 1, 2, -1, -4]
Test 1 Output: [[-1, 0, 1], [-1, 2, -1]]

Test 2 Input: nums = [0, 1, 1]
Test 2 Output: []

Test 3 Input: nums = [0, 0, 0]
Test 3 Output: [[0, 0, 0]]

Test 4 Input: nums = [2, -4]
Test 4 Output: []


## 如何優化我寫的 LeetCode 15 (3Sum) Python 解法，以解決因處理重複組合效率過低而導致的 Time Limit Exceeded (TLE) 錯誤？

---

### 問題出在哪裡？

你的程式碼卡在第 315/316 筆測資，問題出在「事後去重」的策略。

你的寫法是：先找出所有可能的組合，最後再用 `set` 和 `sorted()` 來過濾重複項 (`remove_duplicate_arrays`)。
如果遇到極端的測資（例如陣列裡有 3000 個 `0`），你的 `twoSum` 函數會盲目地將所有可能的配對都加進 `triplets` 陣列中。這會導致 `triplets` 在進入去重階段前，就已經膨脹到數百萬甚至數千萬筆資料，直接耗盡運算時間，引發 Time Limit Exceeded。

### 最佳解法：先排序 (Sort) + 雙指標 (Two Pointers)

為了解決 3Sum，業界標準且最高效的作法是捨棄 Hash Set，改用**先排序搭配雙指標夾擠**。這個方法能讓你**在尋找的當下就跳過重複項**，根本不產生多餘的解答。

#### 核心步驟：

1. **先將陣列排序**：這會花費 O(N log N) 的時間，但能讓相同的數字排在一起，方便我們跳過重複項。
2. **固定一個基準點 (`i`)**：從左到右遍歷陣列。如果 `nums[i]` 和前一個數字相同，就直接 `continue` 跳過，避免產生重複的三元組。
3. **雙指標尋找剩餘兩個數**：設定左指標 `left` 在 `i + 1`，右指標 `right` 在陣列最尾端。
* 如果三數之和 **大於 0**，代表數字太大，將右指標左移 (`right -= 1`)。
* 如果三數之和 **小於 0**，代表數字太小，將左指標右移 (`left += 1`)。
* 如果 **等於 0**，找到一組解答！加入結果後，雙指標同時往內縮，並**跳過所有連續重複的數字**。



---

### 優化後的程式碼

```python
def threeSum(nums: list[int]) -> list[list[int]]:
    result = []
    nums.sort()  # 步驟 1: 先排序
    size = len(nums)
    
    for i in range(size - 2):
        # 優化 1: 如果固定的最小值已經大於 0，後面的數字相加不可能等於 0，直接提早結束
        if nums[i] > 0:
            break
            
        # 優化 2: 遇到重複的基準數字，直接跳過 (避免產生重複的 Triplets)
        if i > 0 and nums[i] == nums[i - 1]:
            continue
            
        left, right = i + 1, size - 1
        
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            
            if total > 0:
                right -= 1
            elif total < 0:
                left += 1
            else:
                # 找到符合的組合
                result.append([nums[i], nums[left], nums[right]])
                
                # 優化 3: 跳過左指標的重複數字
                while left < right and nums[left] == nums[left + 1]:
                    left += 1
                
                # 優化 4: 跳過右指標的重複數字
                while left < right and nums[right] == nums[right - 1]:
                    right -= 1
                    
                # 同時往內移動尋找下一組可能的配對
                left += 1
                right -= 1
                
    return result

```

### 為什麼這樣改會過？

* **時間複雜度降為嚴格的 O(N²)**：排序需要 O(N log N)，外層迴圈跑 N 次，內層雙指標最多跑 N 次（兩個指標相遇即停止）。總結下來是 O(N²)，且沒有任何額外的 $N^3$ 等級的去重成本。
* **空間複雜度降為 O(1) 或 O(N)**：除了儲存答案的 `result` 陣列外，我們不需要宣告巨大的 Hash Set 來記錄看過的數字，只使用了幾個指標變數（不計排序所需的內建空間）。